In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import csv

def scrape_imdb_full_data():
    options = webdriver.ChromeOptions()
    driver = webdriver.Chrome(options=options)

    url = "https://www.imdb.com/search/title/?title_type=feature&sort=user_rating,desc"
    
    driver.get(url)
    time.sleep(4)

    filename = "imdb_movies_dataset.csv"

    headers = [
        "IMDb_ID","Title","Year","Runtime","Certificate",
        "Rating","Vote_Count","Plot_Summary","IMDb_Link","Poster_URL"
    ]

    # create CSV and write header
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(headers)

    target_clicks = 200
    scraped_ids = set()

    for i in range(target_clicks):

        try:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)

            wait = WebDriverWait(driver, 5)
            load_more_btn = wait.until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "button.ipc-see-more__button"))
            )

            driver.execute_script("arguments[0].click();", load_more_btn)
            print(f"Clicked Load More {i+1}/{target_clicks}")
            time.sleep(3)

        except:
            print("No more Load More button")
            break

        # ---------- EXTRACT DATA AFTER EACH CLICK ----------
        soup = BeautifulSoup(driver.page_source, "html.parser")
        movies = soup.select("li.ipc-metadata-list-summary-item")

        new_rows = []

        for movie in movies:
            try:
                title_tag = movie.select_one("h3.ipc-title__text")
                title = title_tag.text.strip() if title_tag else "N/A"
                if ". " in title:
                    title = title.split(". ",1)[-1]

                meta = movie.select("span.dli-title-metadata-item")
                year = meta[0].text.strip() if len(meta)>0 else "N/A"
                runtime = meta[1].text.strip() if len(meta)>1 else "N/A"
                certificate = meta[2].text.strip() if len(meta)>2 else "N/A"

                rating_tag = movie.select_one("span.ipc-rating-star")
                rating = rating_tag.text.strip().split(" ")[0] if rating_tag else "N/A"

                plot_tag = movie.select_one("div.ipc-html-content-inner-div")
                plot = plot_tag.text.strip() if plot_tag else "N/A"

                link_tag = movie.select_one("a.ipc-title-link-wrapper")

                if link_tag:
                    href = link_tag.get("href")
                    imdb_link = f"https://www.imdb.com{href}"
                    imdb_id = href.split("/")[2]
                else:
                    imdb_link = "N/A"
                    imdb_id = "N/A"

                if imdb_id in scraped_ids:
                    continue

                scraped_ids.add(imdb_id)

                img_tag = movie.select_one("img.ipc-image")
                poster = img_tag.get("src") if img_tag else "N/A"

                new_rows.append([
                    imdb_id,title,year,runtime,certificate,
                    rating,"N/A",plot,imdb_link,poster
                ])

            except:
                continue

        # append new data to CSV
        with open(filename,"a",newline="",encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerows(new_rows)

        print(f"Saved {len(new_rows)} new movies to CSV")

    driver.quit()

if __name__ == "__main__":
    scrape_imdb_full_data()

Clicked Load More 1/200
Saved 100 new movies to CSV
Clicked Load More 2/200
Saved 50 new movies to CSV
Clicked Load More 3/200
Saved 50 new movies to CSV
Clicked Load More 4/200
Saved 50 new movies to CSV
Clicked Load More 5/200
Saved 50 new movies to CSV
Clicked Load More 6/200
Saved 50 new movies to CSV
Clicked Load More 7/200
Saved 50 new movies to CSV
Clicked Load More 8/200
Saved 50 new movies to CSV
Clicked Load More 9/200
Saved 50 new movies to CSV
Clicked Load More 10/200
Saved 50 new movies to CSV
Clicked Load More 11/200
Saved 50 new movies to CSV
Clicked Load More 12/200
Saved 50 new movies to CSV
Clicked Load More 13/200
Saved 50 new movies to CSV
Clicked Load More 14/200
Saved 50 new movies to CSV
Clicked Load More 15/200
Saved 50 new movies to CSV
Clicked Load More 16/200
Saved 50 new movies to CSV
Clicked Load More 17/200
Saved 0 new movies to CSV
Clicked Load More 18/200
Saved 100 new movies to CSV
Clicked Load More 19/200
Saved 0 new movies to CSV
Clicked Load More 20/

In [3]:
import pandas as pd
df = pd.read_csv("imdb_movies_dataset.csv")
df.head()

,IMDb_ID,Title,Year,Runtime,Certificate,Rating,Vote_Count,Plot_Summary,IMDb_Link,Poster_URL
0,tt37161609,Sky,2026,2h 14m,NaN,10 (1.1K),NaN,Modern relationships intertwine with career am...,https://www.imdb.com/title/tt37161609/?ref_=sr...,https://m.media-amazon.com/images/M/MV5BMGMyMD...
1,tt40326325,Karanlik Sokaklar,2026,NaN,NaN,10 (5),NaN,NaN,https://www.imdb.com/title/tt40326325/?ref_=sr...,https://m.media-amazon.com/images/M/MV5BMWMwYm...
2,tt40415811,Saravá Shalom,2026,1h 22m,NaN,10 (6),NaN,A narrative that shows the dialogue between Af...,https://www.imdb.com/title/tt40415811/?ref_=sr...,https://m.media-amazon.com/images/M/MV5BMjk4OG...
3,tt37599251,Big Mountain Soul: Ski Africa,2025,45m,NaN,10 (15),NaN,Join the Big Mountain Soul group on an extraor...,https://www.imdb.com/title/tt37599251/?ref_=sr...,https://m.media-amazon.com/images/M/MV5BOWJiZG...
4,tt39455347,TT for Life,2026,1h 27m,NaN,10 (6),NaN,"TT FOR LIFE plunges into the Isle of Man TT, t...",https://www.imdb.com/title/tt39455347/?ref_=sr...,https://m.media-amazon.com/images/M/MV5BOGUzNj...


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   IMDb_ID       10000 non-null  object 
 1   Title         10000 non-null  object 
 2   Year          9999 non-null   object 
 3   Runtime       7914 non-null   object 
 4   Certificate   778 non-null    object 
 5   Rating        10000 non-null  object 
 6   Vote_Count    0 non-null      float64
 7   Plot_Summary  8406 non-null   object 
 8   IMDb_Link     10000 non-null  object 
 9   Poster_URL    9178 non-null   object 
dtypes: float64(1), object(9)
memory usage: 781.4+ KB


In [5]:
df.tail()

,IMDb_ID,Title,Year,Runtime,Certificate,Rating,Vote_Count,Plot_Summary,IMDb_Link,Poster_URL
9995,tt11187612,Anunnaki - Mensageiros do Vento,2016,1h 30m,NaN,8.6 (42),NaN,Mensageiros do Vento is an Opera Rock cartoon....,https://www.imdb.com/title/tt11187612/?ref_=sr...,https://m.media-amazon.com/images/M/MV5BZTQwZT...
9996,tt15717054,Mahishasur Marddini,2022,2h 6m,NaN,8.6 (41),NaN,Mahishasur Marddini presents a collage of narr...,https://www.imdb.com/title/tt15717054/?ref_=sr...,https://m.media-amazon.com/images/M/MV5BZTgxMj...
9997,tt1262918,Kerberos,2010,NaN,NaN,8.6 (19),NaN,Ex-Con bank robber Mike Finn is trying to go s...,https://www.imdb.com/title/tt1262918/?ref_=sr_...,https://m.media-amazon.com/images/M/MV5BYjUyND...
9998,tt15434512,The Real Angelina,2021,47m,NaN,8.6 (15),NaN,"Angelina Jolie, the admirable Oscar-winning ac...",https://www.imdb.com/title/tt15434512/?ref_=sr...,https://m.media-amazon.com/images/M/MV5BYzczYm...
9999,tt7540724,Tu Bold Mee Cold,2016,2h 16m,NaN,8.6 (12),NaN,Can old-fashioned love survive against 21st ce...,https://www.imdb.com/title/tt7540724/?ref_=sr_...,https://m.media-amazon.com/images/M/MV5BODA2Mz...
